# Import thu vien

In [19]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
# preprocess_input của MobileNetV2 sẽ scale pixel từ [0,255] -> [-1, 1]
# (khác với rescale=1/255 scale về [0,1])
# Quan trọng: phải dùng cùng hàm này ở cả train lẫn app.py

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    EarlyStopping,       # tự dừng khi val_accuracy không tăng nữa
    ReduceLROnPlateau,   # tự giảm learning rate khi bị stuck
    ModelCheckpoint      # tự lưu model tốt nhất
)
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices('GPU')) > 0)

TensorFlow version: 2.21.0
GPU available: False


# Cau hinh tham so

In [20]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bhavikjikadara/dog-and-cat-classification-dataset")
DATA_DIR = os.path.join(path, "PetImages")
print("Path to dataset files:", path)
print("DATA_DIR:", DATA_DIR)
# Quick sanity check: list subfolders (should include 'Cat' and 'Dog')
print("DATA_DIR contents:", sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]))

Path to dataset files: C:\Users\Admin\.cache\kagglehub\datasets\bhavikjikadara\dog-and-cat-classification-dataset\versions\1
DATA_DIR: C:\Users\Admin\.cache\kagglehub\datasets\bhavikjikadara\dog-and-cat-classification-dataset\versions\1\PetImages
DATA_DIR contents: ['Cat', 'Dog']


In [21]:
# ── Các thông số quan trọng ──────────────────────────────────
IMG_SIZE    = 224   # MobileNetV2 được thiết kế cho ảnh 224x224
                    # (thay vì 128x128 cũ — giữ nhiều chi tiết hơn)
BATCH_SIZE  = 32    # Số ảnh xử lý cùng lúc trong 1 lần cập nhật weight
EPOCHS_HEAD = 5    # Số epoch cho giai đoạn 1: chỉ train phần đầu (head)
EPOCHS_FINE = 10    # Số epoch cho giai đoạn 2: fine-tune toàn bộ model (cộng thêm với epochs của head)
NUM_CLASSES = 2    # Số class dogs & cats 

# Chuan bi du lieu

In [22]:
# ------------- tang cuong du lieu -------------
# chi ap dung cho train khong ap dung cho validation

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,  # scale [-1, 1] cho MobileNetV2
    validation_split=0.2,                     # 80% train / 20% validation

    # ── Augmentation: các phép biến đổi ngẫu nhiên ──
    rotation_range=15,         # xoay ảnh tối đa ±15 độ
    zoom_range=0.15,           # phóng to/thu nhỏ tối đa 15%
    width_shift_range=0.1,     # dịch ngang tối đa 10% chiều rộng
    height_shift_range=0.1,    # dịch dọc tối đa 10% chiều cao
    horizontal_flip=True,      # Cho phép lật ngang (hữu ích cho dogs/cats)
    brightness_range=[0.8, 1.2], # thay đổi độ sáng ±20% (giả lập ánh sáng khác nhau)
    shear_range=0.1,           # biến dạng nghiêng nhẹ (shear transformation)
    fill_mode='nearest'        # khi dịch/xoay, điền pixel thiếu bằng pixel gần nhất
)

# Validation chỉ cần preprocessing, không cần augmentation
val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

print("✅ Data generators created")

✅ Data generators created


In [23]:
# ── Load dữ liệu từ thư mục ──
train_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),  # resize tất cả ảnh về 224x224
    batch_size=BATCH_SIZE,
    class_mode='categorical',          # one-hot encoding cho multi-class
    subset='training',
    shuffle=True,                      # xáo trộn thứ tự ảnh mỗi epoch
    seed=42
)

val_generator = val_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,                     # validation không cần shuffle
    seed=42
)

class_names = list(train_generator.class_indices.keys())
print(f"Số class: {len(class_names)}")
print(f"Train samples: {train_generator.samples}")
print(f"Validation samples: {val_generator.samples}")
print("Classes:", class_names)

Found 20000 images belonging to 2 classes.
Found 4998 images belonging to 2 classes.
Số class: 2
Train samples: 20000
Validation samples: 4998
Classes: ['Cat', 'Dog']


# Xay dung mo hinh

In [24]:
# ── Bước 1: Load MobileNetV2 base (không có phần phân loại cuối) ──
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,    # False = bỏ phần Dense phân loại cuối của ImageNet
                          # Ta sẽ thêm phần phân loại riêng cho dogs & cats
    weights='imagenet'    # Load trọng số đã train sẵn trên ImageNet
)

# Đóng băng toàn bộ base_model — phase 1 chỉ train HEAD
# (nếu không đóng băng, weights ImageNet bị phá vỡ khi data ít)
base_model.trainable = False

print(f"MobileNetV2 base: {len(base_model.layers)} layers")
print(f"Trainable params khi frozen: {sum(int(tf.size(w)) for w in base_model.trainable_weights)}")

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
MobileNetV2 base: 154 layers
Trainable params khi frozen: 0


In [25]:
# ── Bước 2: Xây dựng HEAD (phần phân loại cho dogs and cats) ──────────
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

# Chạy qua base MobileNetV2 (không train)
x = base_model(inputs, training=False)
# training=False quan trọng: giữ BatchNorm của base ở inference mode

# GlobalAveragePooling2D: tốt hơn Flatten cho transfer learning
# - Flatten: (7,7,1280) → 62720 features (quá nhiều, dễ overfit)
# - GlobalAvgPool: (7,7,1280) → 1280 features (lấy trung bình mỗi feature map)
x = layers.GlobalAveragePooling2D()(x)

# BatchNormalization: chuẩn hóa output trước khi đưa vào Dense
# giúp training ổn định hơn, giảm phụ thuộc vào learning rate
x = layers.BatchNormalization()(x)

# Dense layer với 256 neurons — phần học đặc trưng cho dogs & cats
# Nhỏ hơn 512 vì GlobalAvgPool đã nén thông tin, không cần quá nhiều neurons
x = layers.Dense(
    256,
    activation='relu',  # ReLU: kích hoạt phi tuyến, giúp học đặc trưng phức tạp
    kernel_regularizer=keras.regularizers.l2(0.001)
    # L2 regularization: phạt weight quá lớn → tránh overfit
    # 0.001 là hệ số phạt, tăng nếu vẫn overfit, giảm nếu underfit
)(x)

# Dropout 0.3: mỗi epoch ngẫu nhiên "tắt" 30% neurons
# Thấp hơn 0.5 vì đây là HEAD nhỏ, không cần dropout mạnh
x = layers.Dropout(0.3)(x)

# Output layer: 2 neurons = dogs & cats
# Softmax: chuyển output thành xác suất, tổng = 1
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    # Adam lr=1e-3 phù hợp cho phase 1 (train HEAD từ đầu)
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           514 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,591,554 (9.89 MB)

 Trainable params: 331,010 (1.26 MB)

 Non-trainable params: 2,260,544 (8.62 MB)

# Callbacks - tu dong toi uu qua trinh training

In [26]:
callbacks = [
    # ── EarlyStopping: dừng sớm nếu model không cải thiện ──
    EarlyStopping(
        monitor='val_accuracy',  # theo dõi validation accuracy
        patience=5,              # chờ 5 epoch liên tiếp không tăng thì dừng
        restore_best_weights=True, # khôi phục weights tốt nhất (không phải epoch cuối)
        verbose=1
    ),

    # ── ReduceLROnPlateau: giảm learning rate tự động ──
    ReduceLROnPlateau(
        monitor='val_loss',  # khi val_loss không giảm
        factor=0.3,          # nhân lr với 0.3 (vd: 0.001 → 0.0003)
        patience=3,          # chờ 3 epoch trước khi giảm
        min_lr=1e-7,         # learning rate tối thiểu, không giảm thêm nữa
        verbose=1
    ),

    # ── ModelCheckpoint: lưu model tốt nhất ──
    ModelCheckpoint(
        filepath='d&c_classification_model_best.keras',  # dùng .keras thay .h5 (format mới)
        monitor='val_accuracy',
        save_best_only=True,              # chỉ lưu khi accuracy tốt hơn lần trước
        verbose=1
    )
]

print("✅ Callbacks ready")

✅ Callbacks ready


# Train HEAD (Phan 1)

In [ ]:
print("= PHASE 1: Training HEAD only =")
print(f"Trainable params: {model.trainable_variables.__len__()}")

history_phase1 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_HEAD,
    callbacks=callbacks,
    verbose=1
)

= PHASE 1: Training HEAD only =
Trainable params: 6
Epoch 1/5
172/625 ━━━━━━━━━━━━━━━━━━━━ 8:16 1s/step - accuracy: 0.9502 - loss: 0.5431

c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9656 - loss: 0.4389
Epoch 1: val_accuracy improved from None to 0.98239, saving model to d&c_classification_model_best.keras

Epoch 1: finished saving model to d&c_classification_model_best.keras
625/625 ━━━━━━━━━━━━━━━━━━━━ 802s 1s/step - accuracy: 0.9740 - loss: 0.3402 - val_accuracy: 0.9824 - val_loss: 0.1991 - learning_rate: 0.0010
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9820 - loss: 0.1740
Epoch 2: val_accuracy improved from 0.98239 to 0.98399, saving model to d&c_classification_model_best.keras

Epoch 2: finished saving model to d&c_classification_model_best.keras
625/625 ━━━━━━━━━━━━━━━━━━━━ 763s 1s/step - accuracy: 0.9822 - loss: 0.1550 - val_accuracy: 0.9840 - val_loss: 0.1180 - learning_rate: 0.0010
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9828 - loss: 0.1104

# Fine tuning (Phan 2)

In [ ]:
# ── Mở băng 40 layer cuối của MobileNetV2 ────────────────────
base_model.trainable = True
total_layers = len(base_model.layers)

# Chỉ train layers từ vị trí (total - 40) trở đi
FINE_TUNE_AT = total_layers - 40
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False  # đóng băng các layer đầu (học đặc trưng cơ bản)
for layer in base_model.layers[FINE_TUNE_AT:]:
    layer.trainable = True   # mở 40 layer cuối để fine-tune

print(f"Total layers in base: {total_layers}")
print(f"Fine-tuning from layer: {FINE_TUNE_AT}")
print(f"Layers being fine-tuned: {total_layers - FINE_TUNE_AT}")

# Compile lại với learning rate nhỏ hơn 100 lần so với phase 1
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    # 1e-5 rất nhỏ: tránh làm hỏng weights ImageNet đã tốt
    # Nếu dùng lr cao ở đây → model sẽ "quên" những gì ImageNet đã học
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("=== PHASE 2: Fine-tuning ===")
history_phase2 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_FINE,
    callbacks=callbacks,
    initial_epoch=len(history_phase1.history['accuracy']),
    # initial_epoch giúp tiếp nối epoch count từ phase 1
    verbose=1
)

# Luu mo hinh

In [ ]:
# Lưu model cuối cùng (cũng có model tốt nhất ở d&c_classification_model_best.keras)
model.save("d&c_classification_model.keras")
print("✅ Đã lưu: d&c_classification_model.keras")
print("✅ Đã lưu: d&c_classification_model_best.keras (model tốt nhất)")

#  Danh gia & Bieu do

In [ ]:
print("Phase 1 keys:", list(history_phase1.history.keys()))
print("Phase 2 keys:", list(history_phase2.history.keys()))

In [ ]:
# Kết hợp history 2 phase để vẽ biểu đồ liên tục
def combine_history(h1, h2):
    combined = {}
    for key in h1.history:
        combined[key] = h1.history[key] + h2.history[key]
    return combined

history_all = combine_history(history_phase1, history_phase2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
ax1.plot(history_all['accuracy'], label='Train Accuracy', color='steelblue')
ax1.plot(history_all['val_accuracy'], label='Val Accuracy', color='orange')
ax1.axvline(x=len(history_phase1.history['accuracy'])-1,
            color='red', linestyle='--', label='Fine-tune start')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss
ax2.plot(history_all['loss'], label='Train Loss', color='steelblue')
ax2.plot(history_all['val_loss'], label='Val Loss', color='orange')
ax2.axvline(x=len(history_phase1.history['loss'])-1,
            color='red', linestyle='--', label='Fine-tune start')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Đánh giá cuối
loss, accuracy = model.evaluate(val_generator, verbose=0)
print(f"\n📈 Final Val Loss: {loss:.4f}")
print(f"📈 Final Val Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")